In [ ]:
# =====================================================
# MODEL TRENDING NOTEBOOK
# Tracks model performance and interpretability over time
# =====================================================

# Install dependencies if needed (for Colab)
!pip install pandas numpy matplotlib scikit-learn -q

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display

# Clone the repo
REPO_URL = "https://github.com/sad19016/myscrapers-sad19016.git"
REPO_NAME = "myscrapers-sad19016"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    # Pull latest changes if already cloned
    !git -C {REPO_NAME} pull

RESULTS_DIR = f"{REPO_NAME}/results"
print(f"Results directory: {RESULTS_DIR}")
print(f"Prediction files found: {len(glob.glob(f'{RESULTS_DIR}/*-preds.csv'))}")
print(f"Importance files found: {len(glob.glob(f'{RESULTS_DIR}/*-permutation_importance.csv'))}")
print(f"PDP files found: {len(glob.glob(f'{RESULTS_DIR}/*-pdp_top3.png'))}")

In [ ]:
def calculate_metrics(df):
    """Calculate MAE, MAPE, RMSE, and Bias from a predictions dataframe."""
    # Drop rows where actual_price is missing
    df = df.dropna(subset=["actual_price", "pred_price"])
    if len(df) == 0:
        return None
    
    actual = df["actual_price"]
    predicted = df["pred_price"]
    
    mae = np.mean(np.abs(actual - predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))
    bias = np.mean(predicted - actual)  # positive = overestimating, negative = underestimating
    
    return {"mae": mae, "mape": mape, "rmse": rmse, "bias": bias}

# Load all prediction files and calculate metrics for each run
metrics_rows = []
pred_files = sorted(glob.glob(f"{RESULTS_DIR}/*-preds.csv"))

for filepath in pred_files:
    filename = os.path.basename(filepath)
    run_id = filename.replace("-preds.csv", "")
    
    try:
        df = pd.read_csv(filepath)
        metrics = calculate_metrics(df)
        if metrics:
            metrics["run_id"] = run_id
            metrics["timestamp"] = pd.to_datetime(run_id, format="%Y%m%d%H")
            metrics["n_predictions"] = len(df)
            metrics_rows.append(metrics)
    except Exception as e:
        print(f"Skipping {filename}: {e}")

metrics_df = pd.DataFrame(metrics_rows).sort_values("timestamp")
print(f"Loaded {len(metrics_df)} runs")
display(metrics_df)

In [ ]:
# Load all permutation importance files
importance_runs = {}
importance_files = sorted(glob.glob(f"{RESULTS_DIR}/*-permutation_importance.csv"))

for filepath in importance_files:
    filename = os.path.basename(filepath)
    run_id = filename.replace("-permutation_importance.csv", "")
    try:
        df = pd.read_csv(filepath)
        importance_runs[run_id] = df
    except Exception as e:
        print(f"Skipping {filename}: {e}")

print(f"Loaded {len(importance_runs)} importance files")

In [ ]:
# =====================================================
# SECTION 3: MODEL ACCURACY DASHBOARD
# =====================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Model Accuracy Over Time", fontsize=16, fontweight="bold")

metrics_to_plot = [
    ("mae", "MAE (Mean Absolute Error)", "blue", "Lower is better"),
    ("mape", "MAPE (Mean Absolute % Error)", "orange", "Lower is better"),
    ("rmse", "RMSE (Root Mean Squared Error)", "red", "Lower is better"),
    ("bias", "Bias (+ = Overestimate, - = Underestimate)", "green", "Closer to 0 is better"),
]

for ax, (metric, title, color, subtitle) in zip(axes.flatten(), metrics_to_plot):
    ax.plot(metrics_df["timestamp"], metrics_df[metric], marker="o", color=color, linewidth=2)
    ax.set_title(f"{title}\n({subtitle})", fontsize=11)
    ax.set_xlabel("Run Date")
    ax.set_ylabel(metric.upper())
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("accuracy_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Accuracy dashboard saved!")

In [ ]:
# Summary statistics table
print("=" * 50)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 50)
summary = metrics_df[["run_id", "mae", "mape", "rmse", "bias", "n_predictions"]].copy()
summary["mae"] = summary["mae"].round(2)
summary["mape"] = summary["mape"].round(2)
summary["rmse"] = summary["rmse"].round(2)
summary["bias"] = summary["bias"].round(2)
display(summary)
print(f"\nBest MAE: ${metrics_df['mae'].min():,.2f} on run {metrics_df.loc[metrics_df['mae'].idxmin(), 'run_id']}")
print(f"Latest MAE: ${metrics_df['mae'].iloc[-1]:,.2f}")

In [ ]:
# =====================================================
# SECTION 4: FEATURE IMPORTANCE TRENDING
# =====================================================

# Plot top 15 features for the latest run
latest_run_id = sorted(importance_runs.keys())[-1]
latest_importance = importance_runs[latest_run_id]

# Get top 15 features by importance
top15 = latest_importance.nlargest(15, "importance_mean")

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top15["feature"], top15["importance_mean"], 
               xerr=top15["importance_std"], 
               color="steelblue", alpha=0.8, capsize=3)
ax.set_title(f"Top 15 Features by Permutation Importance\n(Latest Run: {latest_run_id})", 
             fontsize=13, fontweight="bold")
ax.set_xlabel("Importance (drop in negative MAE when feature is shuffled)")
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("feature_importance_latest.png", dpi=150, bbox_inches="tight")
plt.show()
print("Feature importance chart saved!")

In [ ]:
# Track how top 5 features change across runs
if len(importance_runs) > 1:
    # Get top 5 features from latest run
    top5_features = latest_importance.nlargest(5, "importance_mean")["feature"].tolist()
    
    # Build a DataFrame tracking those features across all runs
    tracking_rows = []
    for run_id, imp_df in sorted(importance_runs.items()):
        for feat in top5_features:
            match = imp_df[imp_df["feature"] == feat]
            if not match.empty:
                tracking_rows.append({
                    "run_id": run_id,
                    "timestamp": pd.to_datetime(run_id, format="%Y%m%d%H"),
                    "feature": feat,
                    "importance_mean": match["importance_mean"].values[0]
                })
    
    tracking_df = pd.DataFrame(tracking_rows)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    for feat in top5_features:
        feat_df = tracking_df[tracking_df["feature"] == feat]
        ax.plot(feat_df["timestamp"], feat_df["importance_mean"], 
                marker="o", linewidth=2, label=feat)
    
    ax.set_title("Top 5 Feature Importance Over Time", fontsize=13, fontweight="bold")
    ax.set_xlabel("Run Date")
    ax.set_ylabel("Importance Mean")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("feature_importance_trending.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Only one run available — feature trending will show once more runs complete!")
    print("Run again tomorrow after 10:45 AM to see trending.")

In [ ]:
# =====================================================
# SECTION 5: PARTIAL DEPENDENCE PLOTS (PDPs)
# =====================================================

# Display the latest PDP image
pdp_files = sorted(glob.glob(f"{RESULTS_DIR}/*-pdp_top3.png"))

if pdp_files:
    latest_pdp = pdp_files[-1]
    latest_pdp_run = os.path.basename(latest_pdp).replace("-pdp_top3.png", "")
    
    print(f"Displaying PDP for latest run: {latest_pdp_run}")
    img = mpimg.imread(latest_pdp)
    fig, ax = plt.subplots(figsize=(15, 5))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"Partial Dependence Plots — Top 3 Numeric Features\n(Run: {latest_pdp_run})", 
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("pdp_latest.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No PDP files found yet!")

In [ ]:
# If multiple runs exist, show how PDPs changed over time
if len(pdp_files) > 1:
    print(f"Showing PDPs across {len(pdp_files)} runs:")
    fig, axes = plt.subplots(len(pdp_files), 1, 
                              figsize=(15, 5 * len(pdp_files)))
    
    # Handle case of only 2 runs
    if len(pdp_files) == 2:
        axes = [axes[0], axes[1]]
    
    for ax, pdp_file in zip(axes, pdp_files):
        run_id = os.path.basename(pdp_file).replace("-pdp_top3.png", "")
        img = mpimg.imread(pdp_file)
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(f"PDPs for Run: {run_id}", fontsize=11, fontweight="bold")
    
    plt.suptitle("PDP Evolution Over Time", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("pdp_trending.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Only one run available — PDP trending will show once more runs complete!")
    print("Run again tomorrow after 10:45 AM to see how PDPs change over time.")

In [ ]:
# =====================================================
# SECTION 6: ADDITIONAL DASHBOARDS
# =====================================================

# Load the latest predictions file
latest_preds_file = sorted(glob.glob(f"{RESULTS_DIR}/*-preds.csv"))[-1]
latest_preds = pd.read_csv(latest_preds_file)
latest_preds = latest_preds.dropna(subset=["actual_price", "pred_price"])

# Price Distribution: Predictions vs Actual
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(latest_preds["actual_price"], bins=30, alpha=0.6, 
        color="steelblue", label="Actual Price", edgecolor="white")
ax.hist(latest_preds["pred_price"], bins=30, alpha=0.6, 
        color="orange", label="Predicted Price", edgecolor="white")

ax.set_title("Price Distribution: Actual vs Predicted\n(Latest Run)", 
             fontsize=13, fontweight="bold")
ax.set_xlabel("Price ($)")
ax.set_ylabel("Count")
ax.legend()
ax.grid(True, alpha=0.3)

# Add mean lines
ax.axvline(latest_preds["actual_price"].mean(), color="blue", 
           linestyle="--", linewidth=2, label=f'Actual Mean: ${latest_preds["actual_price"].mean():,.0f}')
ax.axvline(latest_preds["pred_price"].mean(), color="red", 
           linestyle="--", linewidth=2, label=f'Pred Mean: ${latest_preds["pred_price"].mean():,.0f}')
ax.legend()

plt.tight_layout()
plt.savefig("price_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Actual mean price: ${latest_preds['actual_price'].mean():,.2f}")
print(f"Predicted mean price: ${latest_preds['pred_price'].mean():,.2f}")

In [ ]:
# Residual Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left plot: Actual vs Predicted scatter
residuals = latest_preds["pred_price"] - latest_preds["actual_price"]

axes[0].scatter(latest_preds["actual_price"], latest_preds["pred_price"], 
                alpha=0.5, color="steelblue", edgecolors="white", linewidth=0.5)

# Perfect prediction line
min_val = min(latest_preds["actual_price"].min(), latest_preds["pred_price"].min())
max_val = max(latest_preds["actual_price"].max(), latest_preds["pred_price"].max())
axes[0].plot([min_val, max_val], [min_val, max_val], 
             color="red", linestyle="--", linewidth=2, label="Perfect Prediction")
axes[0].set_title("Actual vs Predicted Price", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Actual Price ($)")
axes[0].set_ylabel("Predicted Price ($)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right plot: Residuals distribution
axes[1].hist(residuals, bins=30, color="salmon", edgecolor="white", alpha=0.8)
axes[1].axvline(0, color="red", linestyle="--", linewidth=2, label="Zero Error")
axes[1].axvline(residuals.mean(), color="blue", linestyle="--", 
                linewidth=2, label=f"Mean Error: ${residuals.mean():,.0f}")
axes[1].set_title("Residual Distribution\n(Predicted - Actual)", 
                  fontsize=13, fontweight="bold")
axes[1].set_xlabel("Residual ($)")
axes[1].set_ylabel("Count")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Residual Analysis — Latest Run", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("residual_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Mean residual: ${residuals.mean():,.2f} (positive = overestimating)")
print(f"Std of residuals: ${residuals.std():,.2f}")

In [ ]:
# Price Range Breakdown
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Define price bins
bins = [0, 5000, 15000, 30000, 50000, float("inf")]
labels = ["Under $5k", "$5k-$15k", "$15k-$30k", "$30k-$50k", "Over $50k"]

latest_preds["price_range"] = pd.cut(latest_preds["actual_price"], 
                                      bins=bins, labels=labels)

# Left plot: Count of listings per price range
range_counts = latest_preds["price_range"].value_counts().reindex(labels)
axes[0].bar(range_counts.index, range_counts.values, 
            color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_title("Number of Listings by Price Range", 
                  fontsize=13, fontweight="bold")
axes[0].set_xlabel("Price Range")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)
axes[0].grid(True, alpha=0.3, axis="y")

# Right plot: MAE per price range
range_mae = latest_preds.groupby("price_range", observed=True).apply(
    lambda x: np.mean(np.abs(x["actual_price"] - x["pred_price"]))
).reindex(labels)

axes[1].bar(range_mae.index, range_mae.values, 
            color="salmon", edgecolor="white", alpha=0.8)
axes[1].set_title("MAE by Price Range\n(How accurate is the model per segment?)", 
                  fontsize=13, fontweight="bold")
axes[1].set_xlabel("Price Range")
axes[1].set_ylabel("MAE ($)")
axes[1].tick_params(axis="x", rotation=45)
axes[1].grid(True, alpha=0.3, axis="y")

plt.suptitle("Price Range Analysis — Latest Run", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("price_range_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

# Print summary
print("Listings and MAE per price range:")
for label in labels:
    count = range_counts.get(label, 0)
    mae = range_mae.get(label, 0)
    print(f"  {label}: {count} listings, MAE = ${mae:,.2f}")

In [ ]:
# Mileage vs Price Scatter
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Load all predictions for trending
all_preds = []
for filepath in sorted(glob.glob(f"{RESULTS_DIR}/*-preds.csv")):
    df = pd.read_csv(filepath)
    run_id = os.path.basename(filepath).replace("-preds.csv", "")
    df["run_id"] = run_id
    all_preds.append(df)

all_preds_df = pd.concat(all_preds, ignore_index=True)
all_preds_df = all_preds_df.dropna(subset=["actual_price"])

# Clean mileage
all_preds_df["mileage_num"] = pd.to_numeric(
    all_preds_df["mileage"].astype(str).str.replace(r"[^\d.]", "", regex=True), 
    errors="coerce"
)
all_preds_df = all_preds_df.dropna(subset=["mileage_num"])

# Filter outliers for cleaner visualization
all_preds_df = all_preds_df[
    (all_preds_df["mileage_num"] < 300000) & 
    (all_preds_df["actual_price"] < 100000)
]

# Left plot: Mileage vs Actual Price
axes[0].scatter(all_preds_df["mileage_num"], all_preds_df["actual_price"], 
                alpha=0.4, color="steelblue", edgecolors="white", linewidth=0.5)
axes[0].set_title("Mileage vs Actual Price\n(All Runs)", 
                  fontsize=13, fontweight="bold")
axes[0].set_xlabel("Mileage (miles)")
axes[0].set_ylabel("Actual Price ($)")
axes[0].grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(all_preds_df["mileage_num"], all_preds_df["actual_price"], 1)
p = np.poly1d(z)
x_line = np.linspace(all_preds_df["mileage_num"].min(), 
                     all_preds_df["mileage_num"].max(), 100)
axes[0].plot(x_line, p(x_line), color="red", linestyle="--", 
             linewidth=2, label="Trend")
axes[0].legend()

# Right plot: Mileage vs Prediction Error
all_preds_df["error"] = all_preds_df["pred_price"] - all_preds_df["actual_price"]
axes[1].scatter(all_preds_df["mileage_num"], all_preds_df["error"], 
                alpha=0.4, color="salmon", edgecolors="white", linewidth=0.5)
axes[1].axhline(0, color="red", linestyle="--", linewidth=2, label="Zero Error")
axes[1].set_title("Mileage vs Prediction Error\n(Does mileage affect accuracy?)", 
                  fontsize=13, fontweight="bold")
axes[1].set_xlabel("Mileage (miles)")
axes[1].set_ylabel("Prediction Error ($)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.suptitle("Mileage Analysis — All Runs", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("mileage_vs_price.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Total listings analyzed: {len(all_preds_df)}")
print(f"Average mileage: {all_preds_df['mileage_num'].mean():,.0f} miles")
print(f"Correlation (mileage vs price): {all_preds_df['mileage_num'].corr(all_preds_df['actual_price']):.3f}")

In [ ]:
# Geographic Distribution
# Load the LLM CSV which has city and state fields
import glob

# Load all LLM prediction files and combine
llm_files = sorted(glob.glob(f"{REPO_NAME}/results/*-preds.csv"))
all_llm = []
for filepath in llm_files:
    df = pd.read_csv(filepath)
    all_llm.append(df)

all_llm_df = pd.concat(all_llm, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left plot: Top 10 States
if "state" in all_llm_df.columns and all_llm_df["state"].notna().any():
    state_counts = (all_llm_df["state"]
                   .str.strip()
                   .str.upper()
                   .value_counts()
                   .head(10))
    
    axes[0].barh(state_counts.index, state_counts.values,
                 color="steelblue", edgecolor="white", alpha=0.8)
    axes[0].set_title("Top 10 States by Listing Count",
                      fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Number of Listings")
    axes[0].set_ylabel("State")
    axes[0].invert_yaxis()
    axes[0].grid(True, alpha=0.3, axis="x")
else:
    axes[0].text(0.5, 0.5, "No state data available yet\n(LLM extractor still populating)", 
                ha="center", va="center", fontsize=12, color="gray")
    axes[0].set_title("Top 10 States by Listing Count",
                      fontsize=13, fontweight="bold")

# Right plot: Top 10 Cities
if "city" in all_llm_df.columns and all_llm_df["city"].notna().any():
    city_counts = (all_llm_df["city"]
                  .str.strip()
                  .str.title()
                  .value_counts()
                  .head(10))
    
    axes[1].barh(city_counts.index, city_counts.values,
                 color="salmon", edgecolor="white", alpha=0.8)
    axes[1].set_title("Top 10 Cities by Listing Count",
                      fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Number of Listings")
    axes[1].set_ylabel("City")
    axes[1].invert_yaxis()
    axes[1].grid(True, alpha=0.3, axis="x")
else:
    axes[1].text(0.5, 0.5, "No city data available yet\n(LLM extractor still populating)", 
                ha="center", va="center", fontsize=12, color="gray")
    axes[1].set_title("Top 10 Cities by Listing Count",
                      fontsize=13, fontweight="bold")

plt.suptitle("Geographic Distribution of Listings — All Runs", 
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("geographic_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# Print summary
print("Top 5 States:")
if "state" in all_llm_df.columns and all_llm_df["state"].notna().any():
    print(state_counts.head(5).to_string())
else:
    print("No state data available yet")

print("\nTop 5 Cities:")
if "city" in all_llm_df.columns and all_llm_df["city"].notna().any():
    print(city_counts.head(5).to_string())
else:
    print("No city data available yet")